# agri-agronomy-core quickstart

Minimal example: load the bundled synthetic sample pack, resolve a
taxon by name, and run one irrigation calculation.

No network access. No external files. Deterministic.

In [ ]:
from pathlib import Path

from agri_core import registry
from agri_core.irrigation import irrigate_field_etc
from agri_core.pack import load_knowledge_set


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found above: ' + str(start))


repo_root = _find_repo_root(Path.cwd())
sample_pack = repo_root / 'src' / 'agri_core' / '_samples' / 'sample_pack'
pack = load_knowledge_set(sample_pack, allow_synthetic=True)

info = registry.meta(pack)
print(f'pack version: {info.pack_version}')
print(f'content hash: {info.content_hash[:24]}...')
print(f'engine:       {info.engine_version}')

In [ ]:
outcome = registry.resolve_taxon(pack, 'Testus annualis')
assert outcome.status == 'ok', f'expected ok, got {outcome.status}'
taxon_id = outcome.taxon_id
print(f'resolved taxon: {taxon_id}')

In [ ]:
result = irrigate_field_etc(
    pack,
    taxon_id,
    phase='midseason',
    et0_mm_day=5.0,
)
assert result.status == 'ok', f'expected ok, got {result.status}'
print(f'net irrigation: {result.value:.3f} {result.unit}')
print(f'fact IDs used:  {result.provenance.fact_ids}')

In [ ]:
# M16 export demo: tool schemas and review sheet from the same Pydantic
# models as the library. Offline-safe; no CLI, no subprocess, no network.
import tempfile
from pathlib import Path

from agri_core.export import (
    ReviewSheetFilters,
    generate_review_sheet,
    generate_tool_schemas,
    write_tool_schemas,
)
from agri_core.pack.loader import _default_dictionary

# Tool schemas: fourteen input-only function-calling tools. Each
# description carries a two-sentence refusal contract; host-only
# parameters (pack, min_status, require_commercial) never appear.
schemas = generate_tool_schemas()
assert schemas["tool_schemas_version"] == "1.0.0"
print(f"tool schemas: {len(schemas['tools'])} tools, "
      f"version {schemas['tool_schemas_version']}")
print(f"  first: {schemas['tools'][0]['name']}")
print(f"  last:  {schemas['tools'][-1]['name']}")

# Review sheet: one row per live fact. Default max_status='reviewed'
# (everything below 'approved'), the set a reviewer usually wants.
csv_text, manifest = generate_review_sheet(
    pack, _default_dictionary(), ReviewSheetFilters()
)
assert manifest["sheet_schema_version"] == "1.0.0"
print(f"review sheet: {manifest['row_count']} rows, "
      f"pack {manifest['pack_name']} {manifest['pack_version']}")
print(f"  csv sha256: {manifest['csv_sha256'][:16]}...")

# Round-trip: write both artifacts to a temporary directory. The real
# CLI writes to dist/exports/ (gitignored); a tempdir keeps the notebook
# self-contained and leaves no side effect.
with tempfile.TemporaryDirectory() as tmp:
    ts_path = write_tool_schemas(Path(tmp) / "tool_schemas.json")
    assert ts_path.is_file()
    print(f"  wrote {ts_path.name} ({ts_path.stat().st_size} bytes)")